# Website Insights Agent — Kaggle Capstone Demo

This notebook demonstrates the Website Insights Agent end-to-end using sample data.
It serves as the submission artifact for the AI Agents: Intensive Vibe Coding Course.

**Agent flow:**
1. Load GA4 + Search Console data (sample CSVs here; S3 in production)
2. Summarize data into a structured prompt context
3. Call Gemini to generate actionable recommendations
4. Display recommendations (email step skipped in notebook mode)


In [ ]:
# Install dependencies (Kaggle kernel)
# !pip install google-generativeai pyyaml -q

In [ ]:
import os
import pandas as pd
import google.generativeai as genai

# Set your Gemini API key (use Kaggle Secrets in production)
# from kaggle_secrets import UserSecretsClient
# os.environ['GEMINI_API_KEY'] = UserSecretsClient().get_secret('GEMINI_API_KEY')

GEMINI_API_KEY = os.environ.get('GEMINI_API_KEY', 'YOUR_KEY_HERE')
genai.configure(api_key=GEMINI_API_KEY)

## Step 1: Load Sample Data

In production these DataFrames are loaded from S3. Here we create them inline.

In [ ]:
# Sample GA4 data
ga4_df = pd.DataFrame({
    'date': ['2024-01-01'] * 5,
    'page_path': ['/home', '/blog/seo-tips', '/services', '/about', '/contact'],
    'sessions': [1200, 850, 400, 200, 150],
    'bounce_rate': [0.42, 0.31, 0.55, 0.60, 0.48],
    'avg_session_duration': [185, 320, 95, 78, 62],
})

# Sample Search Console data
sc_df = pd.DataFrame({
    'query': ['seo tips 2024', 'local seo guide', 'how to rank on google',
              'seo checklist', 'google search console tips'],
    'page': ['/blog/seo-tips'] * 5,
    'clicks': [340, 210, 180, 95, 60],
    'impressions': [4200, 3100, 5800, 1200, 900],
    'ctr': [0.081, 0.068, 0.031, 0.079, 0.067],
    'position': [2.1, 3.8, 8.4, 4.2, 5.9],
})

print('GA4 data:')
display(ga4_df)
print('\nSearch Console data:')
display(sc_df)

## Step 2: Summarize Data (Analyzer Tool)

In [ ]:
import sys
sys.path.append('..')

from agent.tools.analyzer import build_analysis_context

context = build_analysis_context(ga4_df, sc_df, site_url='https://example.com')
print(context)

## Step 3: Agent Loop — Call Gemini

In [ ]:
SYSTEM_PROMPT = """You are a website performance analyst. You will be given
traffic and search data for a website. Your job is to:
1. Identify the top 3-5 actionable insights from the data.
2. Prioritize recommendations by expected impact.
3. Keep recommendations specific and concrete.
4. Write in plain language suitable for a non-technical site owner.
"""

model = genai.GenerativeModel(
    model_name='gemini-2.0-flash',
    system_instruction=SYSTEM_PROMPT,
    generation_config={'temperature': 0.2, 'max_output_tokens': 2048},
)

user_prompt = f"Here is the latest website data:\n\n{context}\n\nPlease provide your top recommendations."

response = model.generate_content(user_prompt)
recommendations = response.text

print(recommendations)

## Step 4: Output

In production, `agent/tools/email_sender.py` POSTs these recommendations
to an AWS Lambda URL, which triggers SES to deliver them via email.

In [ ]:
from agent.tools.email_sender import format_email_body

email_body = format_email_body('https://example.com', recommendations)
print(email_body)